# EXP_060C — Alternative Combination 2: EfficientNet-B3 + PhoBERT + FiLM + Huber
**Phase 6 | Promising Combination Validation**
Research question: Does efficient CNN + Vietnamese text + FiLM conditioning synergize?
- Image: EfficientNet-B3 | Text: PhoBERT | Fusion: FiLM | Loss: Huber | Seed: 42

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 13415, done.
remote: Counting objects: 100% (286/286), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 13415 (delta 208), reused 188 (delta 113), pack-reused 13129 (from 1)
Receiving objects: 100% (13415/13415), 873.23 MiB | 32.83 MiB/s, done.
Resolving deltas: 100% (446/446), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

total 1420
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 24 06:16 ..
drwxr-xr-x  2 root root 1437696 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_060C_efficientnetb3_phobert_film_huber'

BEST_IMAGE_EXP_ID = 'EXP_020D_efficientnetb3_xlmr_concat_mse'
BEST_TEXT_EXP_ID  = 'EXP_030B_bestimage_phobert_concat_mse'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts: {DRIVE_EXP_PATH}')

Artifacts: /content/drive/MyDrive/SE365/experiments/EXP_060C_efficientnetb3_phobert_film_huber


### STEP 5: Load pretrained weights

In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_TEXT_EXP_ID}/best_model_train_text.pth', './checkpoints/best_model_train_text.pth')
print(f'Loaded text from {BEST_TEXT_EXP_ID}')
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model_train_image.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image from {BEST_IMAGE_EXP_ID}')

Loaded text from EXP_030B_bestimage_phobert_concat_mse
Loaded image from EXP_020D_efficientnetb3_xlmr_concat_mse


### STEP 6: Train

In [ ]:
!python main.py \
  --mode train_fusion \
  --fusion_type film \
  --text_model_name vinai/phobert-base-v2 \
  --image_model_name efficientnet_b3 \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-5 \
  --grad_accum_steps 2 \
  --patience 5 \
  --loss_fn huber \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_060C_efficientnetb3_phobert_film_huber \
  --exp_dir ./experiments

====== MODE: TRAIN_FUSION ======
Using device: cuda
Seed: 42 | Experiment: EXP_060C_efficientnetb3_phobert_film_huber
Loaded timm processor for efficientnet_b3
Loading weights: 100% 197/197 [00:00<00:00, 21253.10it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/content/SE365/Trainer.py:88: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is dep

### STEP 7: Evaluate on Test Set
Evaluate the best model on the unseen test set to report final metrics and generate plots.

In [ ]:
!python test.py \
  --mode train_fusion \
  --fusion_type film \
  --text_model_name vinai/phobert-base-v2 \
  --image_model_name efficientnet_b3 \
  --exp_id $EXP_ID \
  --exp_dir ./experiments \
  --save_path ./experiments/$EXP_ID

====== TESTING: TRAIN_FUSION ======
Device: cuda
Test samples: 600
Loading weights: 100% 197/197 [00:00<00:00, 17339.11it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loaded weights: ./experiments/EXP_060C_efficientnetb3_phobert_film_huber/best_model_train_fusion.pth
/content/SE365/test.py:100: FutureWarning: `torch.cuda.amp.autocast(args...)` is

### STEP 8: Save to Drive + print metrics


In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

# --- VALIDATION METRICS ---
with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results (Validation) ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")

# --- TEST METRICS ---
with open(f'./experiments/{EXP_ID}/test_metrics.json') as f:
    t = json.load(f)

print(f'\n=== {EXP_ID} Results (Test) ===')
print()
print("             MAE      RMSE      R2")
print(f"  food     : {t['mae_food']:.4f}   {t['rmse_food']:.4f}   {t['r2_food']:.4f}")
print(f"  price    : {t['mae_price']:.4f}   {t['rmse_price']:.4f}   {t['r2_price']:.4f}")
print(f"  atmos    : {t['mae_atmos']:.4f}   {t['rmse_atmos']:.4f}   {t['r2_atmos']:.4f}")
print(f"  service  : {t['mae_service']:.4f}   {t['rmse_service']:.4f}   {t['r2_service']:.4f}")
print(f"  overall  : {t['mae_overall']:.4f}   {t['rmse_overall']:.4f}   {t['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {t['mean_mae']:.4f}")
print(f"  aspect_mae : {t['aspect_mae']:.4f}")
print(f"  overall_mae: {t['overall_mae']:.4f}")



=== EXP_060C_efficientnetb3_phobert_film_huber Results (Validation) ===
Loss (val)   : 0.7267

             MAE      RMSE      R2
  food     : 1.1382   1.5304   0.5550
  price    : 1.1812   1.5895   0.4344
  atmos    : 1.1749   1.5450   0.3849
  service  : 1.1985   1.5830   0.5112
  overall  : 0.9354   1.2565   0.6123

  mean_mae   : 1.1256
  aspect_mae : 1.1732
  overall_mae: 0.9354

=== EXP_060C_efficientnetb3_phobert_film_huber Results (Test) ===

             MAE      RMSE      R2
  food     : 1.1080   1.5101   0.5873
  price    : 1.1674   1.5356   0.4381
  atmos    : 1.2107   1.5974   0.3257
  service  : 1.1150   1.5062   0.5217
  overall  : 0.9171   1.2185   0.6227

  mean_mae   : 1.1036
  aspect_mae : 1.1503
  overall_mae: 0.9171
